In [25]:
import polars as pl
import gc
import glob
import os
import numpy as np
import pandas as pd

In [51]:
def partition_device(ip: str, train_ratio: float = 0.7, data_frac: float = 1):
    safe_ip = ip.replace(":", "_").replace(".", "_")
    path = f"C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\device_{safe_ip}.parquet"
    
    lf_full = pl.scan_parquet(path)

    # total rows
    total = lf_full.select(pl.len()).collect().item()
    
    # take only first X% of data (sequential)
    subset_size = int(total * data_frac)
    lf = lf_full.limit(subset_size)

    # now split this subset
    cutoff = int(subset_size * train_ratio)
    
    train_df = (
        lf.with_row_index("_idx")
        .filter(pl.col("_idx") < cutoff)
        .filter(pl.col("detailed-label") == "-")   # benign only
        .drop("_idx")
        .collect()
    )
    
    test_df = (
        lf.with_row_index("_idx")
        .filter(pl.col("_idx") >= cutoff)
        .drop("_idx")
        .collect()
    )
    
    print(f"{ip} — Using {data_frac*100:.1f}% of data ({subset_size:,} rows)")
    print(f"{ip} — Train (benign): {len(train_df):,} rows")
    print(f"{ip} — Test (mixed):   {len(test_df):,} rows")
    
    return train_df, test_df

In [50]:
def df_size_info(df, name="DF"):
    rows = df.height
    cols = df.width
    size_bytes = df.estimated_size()
    size_mb = size_bytes / (1024 ** 2)

    print(f"{name}:")
    print(f"  Rows: {rows:,}")
    print(f"  Columns: {cols}")
    print(f"  Estimated size: {size_mb:.2f} MB\n")

In [52]:
def print_label_distribution(df, col="detailed-label", name="DF"):
    vc = df[col].value_counts().sort("count", descending=True)
    
    total = df.height
    vc = vc.with_columns(
        (pl.col("count") / total * 100).alias("percentage")
    )
    
    print(f"{name} Label Distribution:")
    print(vc)
    print()

In [53]:
train_df,test_df = partition_device("192_168_1_196") 
df_size_info(train_df)
df_size_info(test_df) 

192_168_1_196 — Using 100.0% of data (10,446,132 rows)
192_168_1_196 — Train (benign): 6,403,650 rows
192_168_1_196 — Test (mixed):   3,133,840 rows
DF:
  Rows: 6,403,650
  Columns: 14
  Estimated size: 403.06 MB

DF:
  Rows: 3,133,840
  Columns: 14
  Estimated size: 197.25 MB



In [68]:
train_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\traintestdata\\train_192_168_1_10.parquet"
test_path  = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\traintestdata\\test_192_168_1_10.parquet"

train_df.write_parquet(train_path)
test_df.write_parquet(test_path)

print(f"Train saved → {train_path}")
print(f"Test  saved → {test_path}")

Train saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\traintestdata\train_192_168_1_10.parquet
Test  saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\traintestdata\test_192_168_1_10.parquet


In [54]:
print_label_distribution(test_df)

DF Label Distribution:
shape: (2, 3)
┌────────────────┬─────────┬────────────┐
│ detailed-label ┆ count   ┆ percentage │
│ ---            ┆ ---     ┆ ---        │
│ cat            ┆ u32     ┆ f64        │
╞════════════════╪═════════╪════════════╡
│ -              ┆ 1857084 ┆ 59.259056  │
│ DDoS           ┆ 1276756 ┆ 40.740944  │
└────────────────┴─────────┴────────────┘



In [55]:

path = f"C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\device_192_168_1_196.parquet"
ts_min = (
    pl.scan_parquet(path)
    .select(pl.col("ts").min())
    .collect()
    .item()
)
print(ts_min)

1545402842.863612


In [56]:
CONN_STATES = ["SF", "S0", "S1", "S3", "SH", "SHR", "REJ", "RSTR", "OTH"]
PROTOS      = ["tcp", "udp", "icmp"]
HISTORY_FLAGS = list("ShADadttfFrRwcT^")

def engineer_features(df: pl.DataFrame, ts_min: float) -> np.ndarray:
    """
    Returns a numpy array of shape (n_rows, n_features).
    All features are numerical and scaled.
    """
    result = {}
    
    # ── Temporal ──────────────────────────────────────────────────
    ts_relative = df["ts"] - ts_min
    result["ts_relative"] = np.log1p(ts_relative.to_numpy())
    
    # ── Log-transformed numerics (handle 0s and nulls) ───────────
    for col in ["duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts"]:
        vals = df[col].fill_null(0).fill_nan(0).to_numpy()
        result[f"{col}_log1p"] = np.log1p(vals)
    
    # ── Ratio features ────────────────────────────────────────────
    orig = df["orig_bytes"].fill_null(0).fill_nan(0).to_numpy()
    resp = df["resp_bytes"].fill_null(0).fill_nan(0).to_numpy()
    result["bytes_ratio"] = resp / (orig + 1)
    
    pkts_o = df["orig_pkts"].fill_null(0).fill_nan(0).to_numpy()
    pkts_r = df["resp_pkts"].fill_null(0).fill_nan(0).to_numpy()
    result["pkts_ratio"] = pkts_r / (pkts_o + 1)
    
    # ── Exact zero flags ─────────────────────────────────────────
    dur = df["duration"].fill_null(0).fill_nan(0).to_numpy()
    result["is_exact_zero_duration"] = (dur == 0.0).astype(np.float32)
    result["is_zero_bytes"]          = ((orig == 0) & (resp == 0)).astype(np.float32)
    
    # ── Port features ─────────────────────────────────────────────
    resp_p = df["id.resp_p"].to_numpy()
    result["resp_p_well_known"]  = (resp_p <= 1023).astype(np.float32)
    result["resp_p_registered"]  = ((resp_p > 1023) & (resp_p <= 49151)).astype(np.float32)
    result["resp_p_ephemeral"]   = (resp_p > 49151).astype(np.float32)
    # # Specific attack-relevant ports
    # for port, name in [(6667, "irc"), (37215, "okiru"), (57722, "heartbeat"), (80, "http"), (443, "https"), (53, "dns"), (123, "ntp")]:
    #     result[f"resp_p_{name}"] = (resp_p == port).astype(np.float32)
    
    # ── conn_state one-hot ────────────────────────────────────────
    conn = df["conn_state"].cast(pl.Utf8).to_numpy()
    for state in CONN_STATES:
        result[f"conn_{state}"] = (conn == state).astype(np.float32)
    
    # ── proto one-hot ─────────────────────────────────────────────
    proto = df["proto"].cast(pl.Utf8).to_numpy()
    for p in PROTOS:
        result[f"proto_{p}"] = (proto == p).astype(np.float32)
    
    # ── history features ──────────────────────────────────────────
    history = df["history"].fill_null("").to_numpy()
    result["history_len"] = np.array([len(h) for h in history], dtype=np.float32)
    for flag in HISTORY_FLAGS:
        result[f"hist_{flag}"] = np.array([flag in h for h in history], dtype=np.float32)
    
    # Stack into matrix
    feature_names = list(result.keys())
    feature_matrix = np.column_stack(list(result.values())).astype(np.float32)
    return feature_names, feature_matrix

In [57]:
train_f_names, train_f_matrix = engineer_features(train_df,1545402842.863612)
feature_train_df = pd.DataFrame(train_f_matrix, columns=train_f_names)
print(feature_train_df.head().T)

                               0         1         2         3         4
ts_relative             0.712424  2.304247  2.489020  2.566230  2.566232
duration_log1p          1.792617  0.134223  1.792617  0.004471  0.003956
orig_bytes_log1p        4.369448  3.891820  4.369448  3.891820  3.891820
resp_bytes_log1p        0.000000  3.891820  0.000000  3.891820  3.891820
orig_pkts_log1p         1.098612  0.693147  1.098612  0.693147  0.693147
resp_pkts_log1p         0.000000  0.693147  0.000000  0.693147  0.693147
bytes_ratio             0.000000  0.979592  0.000000  0.979592  0.979592
pkts_ratio              0.000000  0.500000  0.000000  0.500000  0.500000
is_exact_zero_duration  0.000000  0.000000  0.000000  0.000000  0.000000
is_zero_bytes           0.000000  0.000000  0.000000  0.000000  0.000000
resp_p_well_known       1.000000  1.000000  1.000000  1.000000  1.000000
resp_p_registered       0.000000  0.000000  0.000000  0.000000  0.000000
resp_p_ephemeral        0.000000  0.000000  0.00000

In [58]:
print("Min ts_relative:", feature_train_df["ts_relative"].min())
print("Max ts_relative:", feature_train_df["ts_relative"].max())

Min ts_relative: 0.7124242
Max ts_relative: 11.114435


In [59]:
test_f_names, test_f_matrix = engineer_features(test_df,1545402842.863612)
feature_test_df = pd.DataFrame(test_f_matrix, columns=test_f_names)
print(feature_test_df.head().T)
print("Min ts_relative:", feature_test_df["ts_relative"].min())
print("Max ts_relative:", feature_test_df["ts_relative"].max())

                                0          1          2          3          4
ts_relative             11.114435  11.114435  11.114435  11.114435  11.114435
duration_log1p           0.000000   0.000000   0.000000   0.000000   0.000000
orig_bytes_log1p         0.000000   0.000000   0.000000   0.000000   0.000000
resp_bytes_log1p         0.000000   0.000000   0.000000   0.000000   0.000000
orig_pkts_log1p          0.693147   0.693147   0.693147   0.693147   0.693147
resp_pkts_log1p          0.000000   0.000000   0.000000   0.000000   0.000000
bytes_ratio              0.000000   0.000000   0.000000   0.000000   0.000000
pkts_ratio               0.000000   0.000000   0.000000   0.000000   0.000000
is_exact_zero_duration   1.000000   1.000000   1.000000   1.000000   1.000000
is_zero_bytes            1.000000   1.000000   1.000000   1.000000   1.000000
resp_p_well_known        1.000000   1.000000   1.000000   1.000000   1.000000
resp_p_registered        0.000000   0.000000   0.000000   0.0000

In [61]:
print("Original train_df rows:", len(train_df))
print("Original test_df rows:", len(test_df))
print("New train_df rows:", len(train_f_matrix))
print("New test_df rows:", len(test_f_matrix))

Original train_df rows: 6403650
Original test_df rows: 3133840
New train_df rows: 6403650
New test_df rows: 3133840


In [62]:
def create_sequences(features: np.ndarray, window_size: int = 50, step: int = 25) -> np.ndarray:
    """
    Sliding window over the feature matrix.
    window_size: how many consecutive connections per sequence
    step: how many rows to advance between windows (overlap = window_size - step)
    Returns shape: (n_sequences, window_size, n_features)
    """
    sequences = []
    for i in range(0, len(features) - window_size, step):
        sequences.append(features[i : i + window_size])
    return np.array(sequences, dtype=np.float32)

In [63]:
from sklearn.preprocessing import StandardScaler

# Fit ONLY on benign training sequences
scaler = StandardScaler()

# train_features = engineer_features(train_df, ts_min=train_df["ts"].min())
train_seq      = create_sequences(train_f_matrix, window_size=50, step=25)

# Reshape for scaler: (n_seq * window_size, n_features)
n_seq, win, n_feat = train_seq.shape
train_flat = train_seq.reshape(-1, n_feat)
scaler.fit(train_flat)
train_seq_scaled = scaler.transform(train_flat).reshape(n_seq, win, n_feat)

# Apply same scaler to test (don't refit)
# test_features = engineer_features(test_df, ts_min=train_df["ts"].min())  # use train ts_min
test_seq      = create_sequences(test_f_matrix, window_size=50, step=25)
n_seq_t, win, n_feat = test_seq.shape
test_seq_scaled = scaler.transform(test_seq.reshape(-1, n_feat)).reshape(n_seq_t, win, n_feat)

In [66]:


# ── Helper to convert sequences to DataFrame ──
def seq_to_parquet(seq_array, file_path):
    """
    seq_array: numpy array of shape (n_seq, window_size, n_features)
    Flattens each sequence into a single row: window_size * n_features columns
    """
    n_seq, win, n_feat = seq_array.shape
    # flatten each sequence
    flat_seq = seq_array.reshape(n_seq, win * n_feat)
    
    # generate column names: feat0_t0, feat1_t0, ..., featN_tW
    columns = []
    for t in range(win):
        for f in range(n_feat):
            columns.append(f"f{f}_t{t}")
    
    df = pd.DataFrame(flat_seq, columns=columns)
    df.to_parquet(file_path, index=False)
    print(f"Saved {file_path} ({df.shape[0]} rows, {df.shape[1]} columns)")

# ── Save train ──
seq_to_parquet(train_seq_scaled, "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\staging\\train_sequences_192_168_1_196.parquet")

# ── Save test ──
seq_to_parquet(test_seq_scaled, "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\staging\\test_sequences_192_168_1_196.parquet")

Saved C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\staging\train_sequences_192_168_1_196.parquet (256144 rows, 2050 columns)
Saved C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\staging\test_sequences_192_168_1_196.parquet (125352 rows, 2050 columns)
